[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ManzarIMalik/in2stem-placement/blob/main/Intro_to_Data_Science_with_Python.ipynb)

# What is Data Science

Data science is the business of turning raw data into decisions. It borrows from three places at once: **statistics** (what can this data honestly tell me?), **computer science** (how do I handle it at scale?), and **domain knowledge** (what would actually be surprising here?).

That third one is the one people skip, and it is the one that matters most. A chart is easy to make. Knowing whether it means anything is the hard part - and it is the skill this notebook is really about.

By the end you will have taken **48,895 real New York Airbnb listings** the whole way: loading them, cleaning them, exploring them, visualising them, and then - the important bit - working out which of your findings are real and which are noise wearing a convincing hat. Then you will do it again on a second, completely different dataset of **114,000 Spotify tracks**, to prove the method travels.

**A warning up front.** This notebook will show you a finding that is real and survives every test we throw at it. It will then show you two findings that look every bit as convincing and are worth nothing - one built on a single listing, one on a model that was quietly reading the answers. Producing confident nonsense is the most common way data science goes wrong, and the only reliable defence is knowing what it looks like from the inside.

## Why Python for Data Science

* **It is readable.** You spend your time on the problem, not on the language.
* **The libraries are already there.** NumPy and Pandas do the heavy lifting; Matplotlib draws the pictures; scikit-learn does the machine learning. All of them are in Colab already.
* **Everyone else uses it.** When you get stuck at 2am, somebody has already asked your question.

The three libraries in this notebook stack on top of each other. **NumPy** is the foundation - fast arrays of numbers. **Pandas** is built on NumPy and adds labels, so your columns have names instead of index positions. **Matplotlib** draws whatever the other two produce. Learn them in that order and each one explains the next.

## The Data Science Workflow

Nearly every project walks through the same steps:

1. **Collect** the data - from a file, an API, a database, a scraper.
2. **Clean** it - handle missing values, wrong types, duplicates. This genuinely is most of the job.
3. **Explore** it (EDA) - look at it before you model it. Find the shape, the gaps, and the surprises.
4. **Model** it - build something that predicts or classifies.
5. **Evaluate** it - and be honest, especially when the honest answer is "this works less well than it appears".
6. **Communicate** it - a finding nobody understands is not a finding.

This notebook follows those steps in order, and spends most of its length on steps 2 and 3 - because that is where the real work and nearly all the mistakes live. Steps 4 and 5 come at the end, and step 5 is where we catch a model that looked twice as good as it was.

## Other Tools for Data Science

Python is not the only way to do this, and for a lot of business reporting it is not even the fastest way:

* [Power BI](https://www.microsoft.com/en-us/power-platform/products/power-bi) - Microsoft's dashboard tool, everywhere in industry.
* [Tableau](https://www.tableau.com/) - the other big dashboard tool.
* [Google Looker Studio](https://lookerstudio.google.com/) - free, browser-based, connects straight to Google Sheets.
* **Excel / Google Sheets** - genuinely fine for small data. Do not let anyone make you feel bad about it.

The reason to reach for Python instead: **repeatability**. A notebook re-runs from top to bottom and produces the same answer. A spreadsheet someone hand-edited for three months does not.

In [ ]:
# Colab already ships with almost everything this notebook needs.
import sys
import numpy as np
import pandas as pd
import matplotlib
import sklearn

print("Python version:      ", sys.version.split()[0])
print("NumPy version:       ", np.__version__)
print("pandas version:      ", pd.__version__)
print("Matplotlib version:  ", matplotlib.__version__)
print("scikit-learn version:", sklearn.__version__)
print("Ready to go.")

# Numerical Python (NumPy)

NumPy is the foundation the rest of the scientific Python world is built on. Its one big idea is the **`ndarray`**: a grid of numbers, all of the same type, stored in one continuous block of memory.

That "all the same type" restriction sounds like a limitation. It is actually the whole point - it is what makes NumPy fast, and we will measure exactly how much in a moment.

In [ ]:
import numpy as np

# A 1-dimensional array - like a list, but not a list.
a = np.array([1, 2, 3, 4, 5])
print("1D array:", a)
print("type:    ", type(a))

# A 2-dimensional array - a list of lists becomes a grid.
b = np.array([[1, 2, 3],
              [4, 5, 6]])
print()
print("2D array:")
print(b)

Typing out every number gets old fast. NumPy has constructors for the arrays you actually need.

In [ ]:
# Filled with a constant
print("zeros((2, 3)):")
print(np.zeros((2, 3)))

print("\nones((2, 4)):")
print(np.ones((2, 4)))

# arange(start, stop, step) - like range(), but an array. Stop is NOT included.
print("\narange(0, 10, 2):", np.arange(0, 10, 2))

# linspace(start, stop, n) - n evenly spaced points. Stop IS included.
# This trips everyone up: arange excludes the end, linspace includes it.
print("linspace(0, 1, 5):", np.linspace(0, 1, 5))

# Random numbers, with a seed so you get the same "random" numbers every run.
rng = np.random.default_rng(seed=42)
print("\nrandom integers:", rng.integers(1, 7, size=5))   # five dice rolls

### Array Attributes

Every array can describe itself. When something breaks in NumPy, the answer is almost always in `.shape`.

* `ndim` - how many dimensions.
* `shape` - the size along each dimension, as a tuple.
* `size` - the total number of elements.
* `dtype` - what type the elements are.

In [ ]:
arr = np.array([[1, 2, 3],
                [4, 5, 6]])

print("Number of dimensions:", arr.ndim)    # 2
print("Shape of the array:  ", arr.shape)   # (2, 3) -> 2 rows, 3 columns
print("Size of the array:   ", arr.size)    # 6 -> 2 * 3
print("Data type:           ", arr.dtype)   # int64

# Watch what "all the same type" means in practice:
mixed = np.array([1, 2, 3.5])
print()
print("np.array([1, 2, 3.5]) ->", mixed, "with dtype", mixed.dtype)
print("The integers were quietly converted to floats to keep one type for all.")

## Why NumPy Is Fast

Textbooks assert that NumPy is faster than plain Python lists. Let's not take that on trust - let's time it.

The key idea is **vectorization**. When you write `a * 2` on a NumPy array, the loop still happens, but it happens down in compiled C code over a tight block of memory, instead of in the Python interpreter one object at a time.

In [ ]:
import time

size = 1_000_000
python_list = list(range(size))
numpy_array = np.arange(size)

# The Python way: loop, one element at a time.
start = time.perf_counter()
result_list = [x * 2 for x in python_list]
python_time = time.perf_counter() - start

# The NumPy way: describe what you want to the whole array at once.
start = time.perf_counter()
result_array = numpy_array * 2
numpy_time = time.perf_counter() - start

print(f"Python list: {python_time * 1000:7.2f} ms")
print(f"NumPy array: {numpy_time * 1000:7.2f} ms")
print(f"NumPy is roughly {python_time / numpy_time:.0f}x faster here.")
print()
print("Same answer either way:", result_list[:3], "vs", result_array[:3])

The exact multiple changes every run - it depends on what else the machine is doing - but the order of magnitude is the point. This is why the whole ecosystem sits on NumPy.

There is a second reason, and it matters just as much for readability: `numpy_array * 2` says *what you want*. The list comprehension says *how to get it*. Once you are doing real work, the first one is much harder to get wrong.

## Indexing, Slicing, and Boolean Masks

Indexing an array works like indexing a list, with one big addition: you can index with a **condition**. This is how nearly all real filtering gets done, in both NumPy and Pandas, so it is worth getting comfortable with here where the arrays are small.

In [ ]:
data = np.array([12, 45, 7, 88, 23, 61, 4])

# Ordinary indexing and slicing, exactly like lists
print("First element:  ", data[0])
print("Last element:   ", data[-1])
print("Slice [1:4]:    ", data[1:4])

# A comparison on an array gives you an array of True/False - one per element.
mask = data > 20
print()
print("data > 20 ->", mask)

# Feed that back in as an index, and you get only the elements where it was True.
print("data[data > 20] ->", data[data > 20])

# Which is why counting is just a sum - True counts as 1.
print("How many are over 20?", (data > 20).sum())

For 2D arrays you index with `[row, column]`. A `:` means "everything along this dimension".

In [ ]:
grid = np.array([[1,  2,  3,  4],
                 [5,  6,  7,  8],
                 [9, 10, 11, 12]])

print("Single element [1, 2]:", grid[1, 2])   # row 1, column 2 -> 7
print("Whole row [0, :]:     ", grid[0, :])   # -> [1 2 3 4]
print("Whole column [:, 1]:  ", grid[:, 1])   # -> [2 6 10]
print("Sub-grid [0:2, 1:3]:")
print(grid[0:2, 1:3])

## Aggregations and `axis`

`sum`, `mean`, `min`, `max` and friends collapse an array down to a summary. On a 2D array you get to choose *which direction* to collapse, using `axis`.

The rule that makes `axis` finally click: **`axis` is the dimension that disappears.**

* `axis=0` collapses the rows, leaving one value per column.
* `axis=1` collapses the columns, leaving one value per row.

In [ ]:
grid = np.array([[1,  2,  3,  4],
                 [5,  6,  7,  8],
                 [9, 10, 11, 12]])

print("grid.shape:", grid.shape)             # (3, 4)
print()
print("Everything:      grid.sum()       ->", grid.sum())
print("Down the rows:   grid.sum(axis=0) ->", grid.sum(axis=0), " shape", grid.sum(axis=0).shape)
print("Across the cols: grid.sum(axis=1) ->", grid.sum(axis=1), " shape", grid.sum(axis=1).shape)
print()
print("Note the shapes: (3, 4) with axis=0 gone leaves (4,). With axis=1 gone leaves (3,).")
print()
print("mean:", grid.mean(), " min:", grid.min(), " max:", grid.max())

**Try it yourself:** `scores` below is 4 students by 3 tests. Work out (a) each student's average across their tests, and (b) each test's average across the students. One of them is `axis=0` and one is `axis=1` - use the "the axis disappears" rule to decide which, then check the shape of your answer to confirm.

In [ ]:
scores = np.array([[80, 72, 91],    # student 0
                   [65, 70, 68],    # student 1
                   [90, 95, 99],    # student 2
                   [50, 61, 58]])   # student 3

print("shape:", scores.shape, "-> 4 students, 3 tests")

# TODO: average per student. Should give 4 numbers.
# print("Per student:", scores.mean(axis=?))

# TODO: average per test. Should give 3 numbers.
# print("Per test:   ", scores.mean(axis=?))

**Check yourself** (open once you have committed to an answer):

<details>
<summary><b>Show the answer</b></summary>

`axis=1` gives the per-student averages, `axis=0` gives the per-test averages.

```
scores.mean(axis=1)  ->  [81.0, 67.67, 94.67, 56.33]   4 numbers, one per student
scores.mean(axis=0)  ->  [71.25, 74.5, 79.0]           3 numbers, one per test
```

If you had to guess, you are not alone - this is the single most-confused thing in NumPy, and the confusion comes from reading `axis` as "the direction I want to average along". Try the rule from above instead: **axis is the dimension that disappears.**

`scores` has shape `(4, 3)` - 4 students, 3 tests. Averaging with `axis=1` destroys dimension 1, the one of size 3 (the tests), leaving 4 numbers: one per student. Averaging with `axis=0` destroys dimension 0, the one of size 4 (the students), leaving 3 numbers: one per test.

**Check the shape of what comes back, every time.** Four numbers means you averaged over tests; three means you averaged over students. Getting this backwards will not raise an error - it will hand you a confident, plausible, wrong answer, which is a far worse outcome than a crash. Note that this only works as a check because 4 and 3 are different: on a square array both give the same shape and nothing will save you but reading it properly.

</details>

# Pandas

NumPy is fast but anonymous - `grid[:, 1]` gives you the second column and tells you nothing about what it means. Real data has names, mixed types, and gaps.

**Pandas** is NumPy with labels bolted on. It gives you two structures:

* **`Series`** - a 1D column of values with an index.
* **`DataFrame`** - a 2D table of Series sharing an index. A spreadsheet, or a SQL table.

Underneath, it is still NumPy doing the arithmetic - which is why everything you just learned about vectorization and boolean masks carries straight over.

The next few cells build the structures by hand. After that we stop inventing data and load 48,895 real listings, and everything gets more interesting.

In [ ]:
import pandas as pd
import numpy as np

# A Series from a list. Note np.nan - pandas' marker for "missing".
s = pd.Series([1, 3, 5, np.nan, 6, 8])
print(s)
print()
print("dtype is float64, not int64 - because np.nan is a float, so the whole column became float.")

The index is the thing that makes a Series more than an array. By default it is 0, 1, 2..., but it can be anything - and then you look values up by *name*.

In [ ]:
s = pd.Series([0.25, 0.5, 0.75, 1.0],
              index=['a', 'b', 'c', 'd'])
print(s)
print()

# Look up by label
print("s['b'] ->", s['b'])

# Slice by label - and note the surprise: unlike Python slicing,
# label slicing INCLUDES the endpoint.
print("s['a':'c'] ->", list(s['a':'c']), "  <- 'c' is included!")

# Slice by position - excludes the endpoint, like normal Python.
print("s[0:2]    ->", list(s[0:2]), "        <- position 2 is not")

A **DataFrame** is what you will spend nearly all your time in. Each column is a Series, and columns can have different types.

In [ ]:
data = {'Name': ['Alice', 'Bob', 'Charlie', 'David'],
        'Age': [25, 30, 35, 40],
        'City': ['New York', 'Los Angeles', 'Chicago', 'Houston']}

df_demo = pd.DataFrame(data)
print(df_demo)
print()
print("dtypes - each column has its own type:")
print(df_demo.dtypes)

# Data Analysis - The Art of Storytelling

Everything so far has been dataset-agnostic: arrays, tables, syntax. Now we point it at something real and stay there for a while, because **analysis is the bulk of the job**. Modelling comes at the end, and it is shorter than you expect.

Analysis is storytelling, and that is not a compliment - it is the warning. A dataset does not hand you a story; you build one out of it, choosing what to group by, what to plot, what to leave out. Every one of those choices makes the story clearer, and every one of them is also a chance to tell a story that is not true. The craft is holding both at once: **make it a story worth reading, then try your hardest to prove it wrong.**

The dataset is **NYC Airbnb Open Data**: 48,895 listings scraped from Airbnb's public pages, with location, room type, price, availability and review counts.

This is real data, and real data is imperfect. It has missing values in four columns. It has listings priced at $0, which cannot be right. It has a "minimum nights" of 1250, which is three and a half years and is not a short-term rental. None of that is a flaw in the exercise - **that is the exercise**. A dataset that arrives clean is a dataset somebody already made decisions about, and you did not get to see them.

## Loading the Data

Two practical habits are baked into the loader below, and they are worth more than they look.

**Wrap the download in `try`/`except`.** Networks fail. Hugging Face rate-limits. A URL that worked last term gets moved. If you do not catch that, your notebook dies with a wall of red traceback and a student who thinks they broke Python.

**Cache the file.** Download once, reuse it. Re-downloading 4MB every time you re-run a cell is rude to the host and slow for you.

We load the **parquet** export rather than raw CSV. Parquet is columnar and typed - it is smaller, faster, and it remembers that a number is a number, so you skip a whole category of "why is my price column text?" bugs.

In [ ]:
import os
import time
import pandas as pd

# Seaborn is preinstalled in Colab. Install it if we are somewhere else.
try:
    import seaborn as sns
except ImportError:
    print("seaborn not found - installing...")
    !pip install -q seaborn
    import seaborn as sns

print("seaborn version:", sns.__version__)


def load_hf_dataset(repo, filename, retries=3):
    """Download a Hugging Face dataset's parquet export, with caching and retries.

    Returns a DataFrame, or None if every attempt failed.
    """
    # Cached already? Then we are done - do not hit the network at all.
    if os.path.exists(filename) and os.path.getsize(filename) > 1000:
        print(f"Using cached {filename}")
        return pd.read_parquet(filename)

    url = f"https://huggingface.co/api/datasets/{repo}/parquet/default/train/0.parquet"

    for attempt in range(1, retries + 1):
        try:
            df = pd.read_parquet(url)
            df.to_parquet(filename)          # cache for next time
            print(f"Downloaded {filename}: {df.shape[0]:,} rows x {df.shape[1]} columns")
            return df
        except Exception as e:
            print(f"  attempt {attempt}/{retries} failed ({type(e).__name__})")
            if attempt < retries:
                time.sleep(3)

    print(f"\nCould not download {repo}.")
    print("Check your internet connection, or download the file manually from:")
    print(f"  https://huggingface.co/datasets/{repo}")
    return None

In [ ]:
airbnb = load_hf_dataset("gradio/NYC-Airbnb-Open-Data", "airbnb.parquet")

# Fail loudly and early rather than 20 cells later with a confusing error.
assert airbnb is not None, "Airbnb data failed to load - see the message above."
print("Ready:", airbnb.shape)

### One palette, set once

Every chart in this notebook uses the same colours, defined here and nowhere else. This is not decoration.

Two rules are doing real work:

1. **Colour follows the entity, not its rank.** Manhattan is the same blue whether it sits first or last in a sorted chart. If a colour changes meaning when you re-sort, your reader has to re-learn the legend on every chart.
2. **Sequential data gets one hue, light to dark.** Price on a map is a magnitude, so it gets a single-hue ramp. Rainbow colormaps invent boundaries that are not in the data - your eye reads the yellow-to-green edge as a cliff when the numbers are a smooth slope.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Categorical slots - fixed order, assigned to entities, never recycled.
BLUE, GREEN, PINK, YELLOW, AQUA = "#2a78d6", "#008300", "#e87ba4", "#eda100", "#1baf7a"
ORANGE, VIOLET, RED = "#eb6834", "#4a3aa7", "#e34948"
GREY = "#8a8a86"

# Colour follows the entity: Manhattan is this blue in every chart, at any rank.
BOROUGH_COLORS = {
    "Manhattan": BLUE, "Brooklyn": GREEN, "Queens": PINK,
    "Bronx": YELLOW, "Staten Island": AQUA,
}
ROOM_COLORS = {"Entire home/apt": BLUE, "Private room": GREEN, "Shared room": YELLOW}

# Sequential ramp: one hue, light -> dark. For magnitudes like price.
SEQ = LinearSegmentedColormap.from_list("seq_blue", ["#dbe8f8", "#2a78d6", "#12365f"])
# Diverging ramp: two hues with a NEUTRAL midpoint. For residuals (over/under).
DIV = LinearSegmentedColormap.from_list("div_bo", ["#2a78d6", "#eeeeec", "#eb6834"])

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.edgecolor": "#c9c9c4", "axes.labelcolor": "#3d3d3a",
    "grid.color": "#e6e6e2",          # recessive grid - the data is the ink
    "font.size": 10,
})
print("Palette and chart theme set.")

## The First Look

Four questions, every dataset, every time: **How big is it? What is in it? What does a row look like? What are the numbers doing?**

`.shape`, `.info()`, `.head()` and `.describe()` answer those in that order. Run them before you have an opinion.

In [ ]:
rows, cols = airbnb.shape
print(f"Shape: {airbnb.shape} -> {rows:,} listings, {cols} columns")
print()
for c in airbnb.columns:
    print("  -", c)

In [ ]:
# .head() - what does a single listing actually look like?
airbnb.head()

In [ ]:
# .info() - types, and how many non-null values per column. Read the counts.
airbnb.info()

That `.info()` output is where this dataset stops being a toy.

**Four columns have gaps**, and unlike the tidy datasets in tutorials, the gaps mean different things:

| Column | Missing | What the gap means |
|---|---|---|
| `last_review` | 10,052 | Never been reviewed |
| `reviews_per_month` | 10,052 | Never been reviewed - *the same 10,052 rows* |
| `host_name` | 21 | Genuinely absent - blank or removed |
| `name` | 16 | Genuinely absent - listing had no title |

Notice those first two numbers are identical. That is not a coincidence, and spotting it is the difference between deleting a fifth of your data and understanding it. We come back to this in the cleaning section, because it drives the single most important decision in this notebook.

Also check the `Dtype` column. `price` is `int64` - good. If a column you expect to be numeric shows up as `object`, something is hiding in it: a `$` sign, an `"N/A"`, a stray comma.

In [ ]:
# .describe() - the statistical summary of every numeric column.
airbnb.describe().round(2)

Read the `price` column of that table slowly, because it is telling you three separate things.

* **Minimum is 0.** A free Airbnb does not exist. Somebody's data entry is broken, or the listing is inactive. Either way it is not a price and it must not go into a price model.
* **Maximum is 10,000, but the 75th percentile is 175.** Three quarters of listings are under $175, and the top of the range is fifty-seven times that. A handful of extreme values are dragging the mean.
* **Mean (152.7) is well above the median (106).** That gap is the signature of a **right-skewed** distribution. When mean and median disagree this much, the mean is not describing a typical listing - it is describing a typical listing plus a few penthouses.

`minimum_nights` deserves the same look: the max is **1250 nights**, which is three and a half years. We will come back to that one too.

This is what `.describe()` is for. Not "here are some numbers" - **"here are the three things I now need to investigate."**

In [ ]:
# Missing values are easier to judge as a picture than as a column of counts.
missing = airbnb.isnull().sum()
missing = missing[missing > 0].sort_values()

fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.barh(missing.index, missing.values, color=BLUE, height=0.6)
for bar, v in zip(bars, missing.values):
    # Direct labels: the number is on the bar, so nobody reads it off a gridline.
    ax.text(v + 150, bar.get_y() + bar.get_height() / 2,
            f"{v:,}  ({v / len(airbnb):.1%})", va="center", fontsize=9, color="#3d3d3a")
ax.set_title("Missing values by column")
ax.set_xlabel("Number of missing rows")
ax.set_xlim(0, 13000)
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

print("Rows where BOTH review columns are missing:",
      (airbnb["last_review"].isnull() & airbnb["reviews_per_month"].isnull()).sum())
print("Of those, how many have number_of_reviews == 0? ",
      (airbnb.loc[airbnb["reviews_per_month"].isnull(), "number_of_reviews"] == 0).sum())

There it is, and it settles the question.

**All 10,052 listings with a missing `reviews_per_month` have `number_of_reviews == 0`.** The value is not lost. It was never generated, because the listing has never been reviewed. The true value is **zero reviews per month**, and the dataset simply left the cell blank instead of writing it down.

That single sentence is worth more than any chart in this section, and it decides what we do next.

## Cleaning the Data

The textbook move at this point is:

```python
df = df.dropna()
```

On this dataset that would delete **10,052 listings - 21% of the data** - because one column was blank in a way that actually meant "zero". You would throw away a fifth of New York to avoid typing `fillna(0)`.

Before we do it properly, here is the general toolkit on a small broken table, where you can see every row change.

In [ ]:
import numpy as np

# A deliberately messy table - the kind of thing real data actually looks like.
messy = pd.DataFrame({
    "name":   ["Alice", "Bob", "Charlie", "Bob", "Eve"],
    "age":    [25, np.nan, 35, np.nan, 29],
    "salary": ["50000", "60000", "unknown", "60000", "55000"],
})
print("Original:")
print(messy)
print("\nMissing per column:")
print(messy.isnull().sum())

# 1. Drop exact duplicate rows (Bob appears twice, identically).
messy = messy.drop_duplicates()
print("\nAfter drop_duplicates() ->", len(messy), "rows")

# 2. Force a text column to numeric. errors="coerce" turns "unknown" into NaN
#    instead of raising, so we can decide what to do with it.
messy["salary"] = pd.to_numeric(messy["salary"], errors="coerce")
print("\nAfter to_numeric - note 'unknown' became NaN:")
print(messy)

# 3. Fill the gaps with the median rather than dropping the row.
#    Dropping would cost us Charlie's salary because his AGE was missing.
messy["age"] = messy["age"].fillna(messy["age"].median())
messy["salary"] = messy["salary"].fillna(messy["salary"].median())

print("\nCleaned:")
print(messy)
print("\ndtypes are numeric now, so arithmetic works:")
print(messy.dtypes)

Step 3 is the judgement call, and it is worth being precise about *why* the median was defensible there.

Charlie's salary was `"unknown"`. We do not know it. We filled in the median because the alternative - `dropna()` - would have deleted his name and age too, and a typical value is a less damaging guess than losing the row. **We invented a number, knowingly, and we should remember that we did.**

Now contrast that with our Airbnb column.

**A missing `reviews_per_month` is not unknown. It is zero.** The listing has no reviews; we confirmed that directly. Filling it with the median (`0.72`) would be inventing review activity for 10,052 listings that have never had a single review - fabricating data, then modelling the fabrication.

That is the distinction to carry with you:

* **Missing at random / genuinely unknown** (Charlie's salary) - impute a typical value, and note that you did.
* **Missing because the thing did not happen** (never reviewed) - fill with the value it logically is. Here, `0`.
* **Missing because something upstream is broken** - go and fix the upstream thing, or drop with your eyes open.

Same syntax, three different decisions. `dropna()` cannot tell them apart. You can.

In [ ]:
# --- Cleaning the Airbnb data: one decision at a time, each one justified. ---
clean = airbnb.copy()
before = len(clean)

# 1. Missing reviews_per_month means ZERO reviews, not an unknown value.
clean["reviews_per_month"] = clean["reviews_per_month"].fillna(0)

# 2. last_review is a date that does not exist for never-reviewed listings.
#    We leave it missing on purpose - inventing a review date would be a lie.
clean["last_review"] = pd.to_datetime(clean["last_review"], errors="coerce")

# 3. price == 0 is impossible. 11 listings. Remove them.
zero_price = (clean["price"] == 0).sum()
clean = clean.loc[clean["price"] > 0]

# 4. Extreme prices: keep the analysis about the market, not about 239 outliers.
#    This is a CHOICE, not a rule - we state the threshold so it can be argued with.
expensive = (clean["price"] > 1000).sum()
clean = clean.loc[clean["price"] <= 1000]

# 5. minimum_nights above a year is not a short-term rental.
long_stay = (clean["minimum_nights"] > 365).sum()
clean = clean.loc[clean["minimum_nights"] <= 365]

clean = clean.reset_index(drop=True)

print(f"Removed {zero_price} listings priced at $0")
print(f"Removed {expensive} listings priced over $1000")
print(f"Removed {long_stay} listings with minimum_nights > 365")
print(f"\n{before:,} rows -> {len(clean):,} rows  ({before - len(clean)} removed, "
      f"{(before - len(clean)) / before:.2%})")
print("Missing reviews_per_month now:", clean["reviews_per_month"].isnull().sum())

# If the upstream dataset ever changes shape, fail here rather than silently
# rewriting every number quoted in the text below.
assert len(clean) == 48631, f"Expected 48,631 clean rows, got {len(clean):,}"

**264 rows removed - about half a percent.** Compare that with the 10,052 rows (21%) that a reflexive `dropna()` would have cost, and for a worse reason.

That `assert` on the last line is a habit worth stealing. Every number in the markdown below was read off a real run of this notebook. If the upstream dataset changes, the assert fails immediately and loudly, instead of letting the prose quietly become fiction while the code still runs.

## Selecting and Filtering

Everything you learned about NumPy boolean masks works here, except the columns have names now.

In [ ]:
# One column -> a Series. A list of columns -> a DataFrame (note the double brackets).
print("One column: ", type(clean["price"]).__name__)
print("Two columns:", type(clean[["price", "room_type"]]).__name__)
print()

# Boolean filtering - the NumPy idea, with names.
cheap_manhattan = clean.loc[(clean["neighbourhood_group"] == "Manhattan") & (clean["price"] < 100)]
print("Manhattan listings under $100:", f"{len(cheap_manhattan):,}")

# Combine with & (and) / | (or). The brackets around each condition are REQUIRED,
# because & binds tighter than == in Python.
entire_cheap = clean.loc[(clean["room_type"] == "Entire home/apt") & (clean["price"] < 100)]
print("Entire homes under $100:      ", f"{len(entire_cheap):,}")

print()
print(cheap_manhattan[["neighbourhood", "room_type", "price", "number_of_reviews"]].head())

`.loc` and `.iloc` are how you reach rows. The distinction catches everyone once:

* **`.loc[]`** - by **label**. The endpoint is **included**.
* **`.iloc[]`** - by **integer position**. The endpoint is **excluded**, like normal Python slicing.

Pick one and stay in it for a whole expression. Mixing them in a single line - `df.iloc[0][["price"]]` - happens to work, and teaches your fingers a habit that breaks the moment your index stops being 0, 1, 2.

In [ ]:
cols = ["neighbourhood_group", "room_type", "price"]

# .loc - by label, all the way through. One consistent indexer.
print("First row, by label:")
print(clean.loc[0, cols])

print("\nRows 0-2 by label (endpoint included):")
print(clean.loc[0:2, cols])

# .iloc - by position, all the way through.
print("\nRows 0-2 by position (endpoint excluded):")
print(clean.iloc[0:3][cols])

print("\n.loc[0:2]  gives", len(clean.loc[0:2]), "rows (endpoint included)")
print(".iloc[0:2] gives", len(clean.iloc[0:2]), "rows (endpoint excluded)")

## Grouping and Aggregating

`groupby` is the workhorse of analysis. One idea, three steps - **split, apply, combine**:

1. **Split** the rows into groups by some column.
2. **Apply** a summary to each group.
3. **Combine** the answers into a new table.

"Average price per borough" is: split by `neighbourhood_group`, apply `mean` to `price`, combine.

In [ ]:
print("Average price by borough:")
print(clean.groupby("neighbourhood_group")["price"].mean().round(2).sort_values(ascending=False))

print("\nBut a mean on its own is half a story. Add count, median and spread:")
borough_stats = (clean.groupby("neighbourhood_group")["price"]
                 .agg(["count", "mean", "median", "std"]).round(1)
                 .sort_values("mean", ascending=False))
print(borough_stats)

That second table is the one to trust, and the extra columns each do a job.

* **`count`** tells you how much weight the row can bear. Manhattan has 21,482 listings; Staten Island has 371. Those two means are not equally reliable, and a bar chart draws them identically.
* **`median`** is well below `mean` in every borough (Manhattan: 149 vs 178.9). The right skew we saw in `.describe()` is present *inside* every group, not just overall.
* **`std`** is the sanity check. Manhattan's mean is 178.9 with a standard deviation of 133.9 - the spread within the borough is comparable to the differences between boroughs. Hold that thought; it is what the shuffle test later is designed to settle.

The gap between Manhattan ($178.9) and the Bronx ($85.4) is **$93.58**, more than double. That is a big claim. We are going to make it earn its place.

**Try it yourself:** use `groupby` to find the average `price` for each `room_type`, then swap `.mean()` for `.agg(["count", "mean", "median", "std"])`.

Before you run it, commit to a guess: is the room-type gap **bigger or smaller** than the $93.58 borough gap? Write it down, then check.

In [ ]:
# TODO: average price per room_type
# print(clean.groupby(?)[?].mean())

# TODO: now with count, median and std as well.
# print(clean.groupby(?)[?].agg([?]))

**Check yourself** (open once you have written down your guess):

<details>
<summary><b>Show the answer</b></summary>

```
                 count   mean  median
Entire home/apt  25207  194.6   160.0
Private room     22269   84.8    70.0
Shared room       1155   67.7    45.0
```

The room-type gap is **$126.90** (Entire home minus Shared room) - noticeably *bigger* than the $93.58 borough gap. And it should be: an entire apartment and a shared room are different products, whereas two boroughs are the same product in different places.

There is a second thing in that table worth catching. **`Shared room` has only 1,155 listings out of 48,631 - 2.4%.** It is a real category, but it is a small one, and any statistic you compute on it will be shakier than the other two. Small groups are where confident nonsense comes from, and we build a whole section around that below.

</details>

# Visualising

A table hides shape; a chart shows it instantly. Which is exactly why charts are also the easiest way to mislead people, starting with yourself.

Picking one is mostly mechanical:

| You want to show | Use |
|---|---|
| The distribution of one numeric column | **Histogram** or **KDE** |
| The distribution *split by* a category | **Box plot** or **violin** |
| Comparing one value across categories | **Bar chart** |
| The relationship between two numeric columns | **Scatter plot** |
| Two categories against each other | **Heatmap** or **stacked bar** |
| Something changing over time | **Line chart** |

We will use most of them, and interrogate every one.

In [ ]:
# The distribution of price - the single most important chart in this notebook.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(clean["price"], bins=60, color=BLUE, edgecolor="white", linewidth=0.5)
axes[0].axvline(clean["price"].mean(), color=ORANGE, linewidth=2,
                label=f"mean  ${clean['price'].mean():.0f}")
axes[0].axvline(clean["price"].median(), color=VIOLET, linewidth=2,
                label=f"median ${clean['price'].median():.0f}")
axes[0].set_title("Price distribution (linear scale)")
axes[0].set_xlabel("Price per night (USD)"); axes[0].set_ylabel("Number of listings")
axes[0].legend()

# Same data, log x-axis. The skew is a scale problem, not a data problem.
axes[1].hist(clean["price"], bins=60, color=BLUE, edgecolor="white", linewidth=0.5)
axes[1].set_xscale("log")
axes[1].set_title("Price distribution (log scale)")
axes[1].set_xlabel("Price per night (USD, log)"); axes[1].set_ylabel("Number of listings")

plt.tight_layout(); plt.show()

print(f"mean   ${clean['price'].mean():7.2f}")
print(f"median ${clean['price'].median():7.2f}")
print(f"skew    {clean['price'].skew():7.2f}   (0 would be symmetric)")

**Left chart:** a wall at the left and a long thin tail to the right. Most listings are cheap; a few are very expensive. The orange mean sits to the right of the violet median, pulled by that tail - exactly the pattern `.describe()` hinted at.

**Right chart:** the same numbers on a log x-axis, and suddenly it is a clean, almost symmetric hump. Nothing about the data changed. **The skew was partly a property of the ruler, not the thing being measured.**

That is the practical reason to reach for a log scale: prices, incomes, city populations and file sizes are all multiplicative quantities, and multiplicative quantities look normal in log space. The honest caveat is that a log axis also visually shrinks big differences, so it is a tool for *seeing structure*, not for *presenting a headline*.

Either way: **"the average price is $141" was never a useful sentence.** Half of the listings are under $105.

In [ ]:
# A bar chart shows one number per group. A box plot shows the whole distribution.
order = clean.groupby("neighbourhood_group")["price"].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.boxplot(data=clean, x="neighbourhood_group", y="price", order=order, ax=ax,
            hue="neighbourhood_group", hue_order=order, legend=False,
            palette=BOROUGH_COLORS,        # colour follows the borough, not its rank
            showfliers=False, linewidth=1.2, width=0.6)
ax.set_title("Price by borough - the whole distribution, not just the average")
ax.set_xlabel(""); ax.set_ylabel("Price per night (USD)")
plt.tight_layout(); plt.show()

print("Box = middle 50% of listings. Line inside = median. Whiskers = the bulk of the rest.")
print("Outliers hidden (showfliers=False) so the boxes are readable - they still exist.")

This chart does something a bar chart structurally cannot: it shows you **the overlap**.

Manhattan's box sits highest, and its median line is clearly above Brooklyn's. But look at how much vertical space the boxes share. A large fraction of Manhattan listings cost less than a large fraction of Brooklyn listings. The boroughs differ *on average* while overlapping *enormously* case by case.

Both things are true at once, and only one of them survives into a bar chart. "Manhattan is more expensive" is a statement about distributions, not about any particular flat - and if you ever have to defend a finding to someone who has a counter-example ("my Manhattan place was $60"), this is the chart that lets you say "yes, and that is consistent with what I found."

In [ ]:
# Now the bar chart - the same comparison, reduced to one number per borough.
stats = (clean.groupby("neighbourhood_group")["price"]
         .agg(["count", "mean", "std"]).sort_values("mean"))

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(stats.index, stats["mean"],
               color=[BOROUGH_COLORS[b] for b in stats.index], height=0.65)
for bar, (name, row) in zip(bars, stats.iterrows()):
    # Direct labels carry the value AND the sample size - the two things a bar hides.
    ax.text(row["mean"] + 2, bar.get_y() + bar.get_height() / 2,
            f"${row['mean']:.0f}   n={int(row['count']):,}",
            va="center", fontsize=9, color="#3d3d3a")
ax.set_title("Average price by borough")
ax.set_xlabel("Average price per night (USD)")
ax.set_xlim(0, 215)          # starts at zero, on purpose
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

Two deliberate choices in that chart, both of which the defaults would have got wrong.

**The x-axis starts at zero.** Matplotlib would happily have started it at 80, which turns a real-but-moderate difference into a dramatic one. Manhattan is about **2.1x** the Bronx - genuinely a lot - and the chart should show exactly 2.1x, not "the bar is eight times longer."

**Every bar is labelled with its sample size.** Staten Island's average rests on 371 listings; Brooklyn's on 20,035. A bar chart draws both as equally solid rectangles, and that is its central lie. Writing `n=` on the bar is the cheapest fix in data visualisation.

In [ ]:
# Two categories at once: composition of room types within each borough.
comp = pd.crosstab(clean["neighbourhood_group"], clean["room_type"], normalize="index") * 100
comp = comp.loc[order]        # same borough order as the box plot above

fig, ax = plt.subplots(figsize=(9, 4))
left = np.zeros(len(comp))
for room in ["Entire home/apt", "Private room", "Shared room"]:
    ax.barh(comp.index, comp[room], left=left, height=0.6,
            color=ROOM_COLORS[room], label=room, edgecolor="white", linewidth=2)
    for i, (v, l) in enumerate(zip(comp[room], left)):
        if v > 6:             # only label segments with room for text
            ax.text(l + v / 2, i, f"{v:.0f}%", ha="center", va="center",
                    color="white", fontsize=9, fontweight="bold")
    left += comp[room].values

ax.set_title("Room type mix within each borough (% of that borough's listings)")
ax.set_xlabel("Share of listings (%)"); ax.set_xlim(0, 100)
ax.legend(loc="lower right", framealpha=0.95)
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

This chart quietly complicates the borough story, which is what good charts do.

**Manhattan is 61% entire homes; the Bronx is 35%.** So when we say "Manhattan is more expensive", part of that gap is not location at all - it is **product mix**. Manhattan rents more whole apartments, whole apartments cost more everywhere, and the borough average inherits that.

This is **confounding**, and it is the reason "average price by X" is almost never the end of an analysis. The honest question is no longer "is Manhattan more expensive?" but "is Manhattan more expensive *for the same kind of room*?" The model we build later gets at that by giving borough and room type to the model together, so each is measured with the other held constant.

In [ ]:
# The hero chart: every listing at its real coordinates, coloured by price.
fig, ax = plt.subplots(figsize=(9, 9))

sc = ax.scatter(clean["longitude"], clean["latitude"],
                c=clean["price"], cmap=SEQ,
                s=2.5, alpha=0.55, linewidths=0,
                vmin=0, vmax=350)        # cap the ramp so the tail does not eat the scale

cbar = plt.colorbar(sc, ax=ax, shrink=0.55, pad=0.02)
cbar.set_label("Price per night (USD)  -  capped at $350 for colour")
ax.set_title("48,631 Airbnb listings, coloured by price\n(nobody drew a map - this is just latitude and longitude)")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_aspect(1.0 / np.cos(np.radians(40.7)))     # correct for NYC's latitude
ax.grid(visible=False)
plt.tight_layout(); plt.show()

Nobody supplied a map. There is no shapefile, no coastline, no borough boundary in that chart - it is 48,631 dots at their latitude and longitude, and **New York City draws itself**, down to Central Park's dark rectangle of absent listings and the gap where the East River runs.

That is the first payoff: geographic columns are worth plotting raw, because human eyes are extremely good at recognising places, and errors show up instantly. A listing in the Atlantic is a data-quality bug you will *see* long before you compute it.

The second payoff is the colour. The dark cluster is Lower and Midtown Manhattan and it fades outward in every direction. This is the borough effect from the bar chart, but far more precise: **price falls off with distance from the centre**, continuously, and it does not care where the borough line is. Waterfront Brooklyn - Williamsburg, right across the river - is darker than northern Manhattan.

So the bar chart was a coarse summary of something real but slightly different. "Manhattan is expensive" is roughly true. "Being close to Midtown is expensive" is closer to the actual mechanism, and you can only see the difference by plotting position instead of grouping by label.

In [ ]:
# Three more distributions worth a look before we start trusting anything.
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))

axes[0].hist(clean["availability_365"], bins=50, color=BLUE, edgecolor="white", linewidth=0.4)
axes[0].set_title("Days available per year"); axes[0].set_xlabel("availability_365")
axes[0].set_ylabel("Listings")

axes[1].hist(clean["number_of_reviews"], bins=50, color=GREEN, edgecolor="white", linewidth=0.4)
axes[1].set_yscale("log")
axes[1].set_title("Number of reviews (log y)"); axes[1].set_xlabel("number_of_reviews")

axes[2].hist(clean.loc[clean["minimum_nights"] <= 30, "minimum_nights"],
             bins=30, color=YELLOW, edgecolor="white", linewidth=0.4)
axes[2].set_title("Minimum nights (capped at 30)"); axes[2].set_xlabel("minimum_nights")

plt.tight_layout(); plt.show()

print("Listings available 0 days a year:", f"{(clean['availability_365'] == 0).sum():,}",
      f"({(clean['availability_365'] == 0).mean():.1%})")
print("Listings with 0 reviews:        ", f"{(clean['number_of_reviews'] == 0).sum():,}",
      f"({(clean['number_of_reviews'] == 0).mean():.1%})")
print("Most common minimum_nights:     ", clean["minimum_nights"].mode()[0])

Every one of those three has a spike that means something, and none of them are noise.

* **A third of listings show `availability_365 == 0`** - over a third. Read literally that says "available zero days a year", which cannot be true of a live listing. It almost certainly means the calendar was full, unset, or the listing was inactive when the data was scraped. **A zero here is not a measurement, it is a shrug**, and treating it as "this place is never available" would be wrong.
* **Reviews are extreme.** The y-axis is logarithmic and it *still* falls off a cliff. 9,911 listings - a fifth of the cleaned data - have never been reviewed, and a handful have hundreds.
* **Minimum nights spikes at 1, 2, 3 and 30.** Those are human choices, not a smooth distribution. The most common value is 1, and the spike at 30 is people positioning against short-term-rental rules - a fact about New York housing law visible in a histogram.

The general habit: **a spike in a histogram is a question, not a feature.** Ask what human process would produce exactly that value, that often.

# When a Chart Does (and Does Not) Lie

We now have a finding: **Manhattan listings average $178.90 against the Bronx's $85.40, a gap of $93.58.**

The chart looks convincing. But so did the chart in every analysis that was ever wrong. Before we act on it, we test it - and the test is the same one regardless of whether the answer turns out to be yes or no.

**The shuffle test.** If borough had *no* effect on price, then the borough labels would be meaningless decoration. So: keep every price exactly as it is, randomly reassign the borough labels, and recompute the gap. Do that a thousand times and you have a picture of **what gaps look like when nothing is going on**. Then check where your real gap sits.

Commit to a guess first. Our earlier warning about small samples applies here too - Staten Island has only 371 listings. Do you expect the real gap to stand out, or to disappear into the noise?

In [ ]:
observed_gap = (clean.groupby("neighbourhood_group")["price"].mean().max() -
                clean.groupby("neighbourhood_group")["price"].mean().min())
print(f"Observed gap (highest borough mean - lowest): ${observed_gap:.2f}")

rng = np.random.default_rng(seed=0)
shuffled_gaps = []
for _ in range(1000):
    fake_labels = rng.permutation(clean["neighbourhood_group"].values)
    means = clean.groupby(fake_labels)["price"].mean()
    shuffled_gaps.append(means.max() - means.min())
shuffled_gaps = np.array(shuffled_gaps)

print(f"\n1000 shuffles with the labels randomised:")
print(f"  average fake gap: ${shuffled_gaps.mean():.2f}")
print(f"  largest fake gap: ${shuffled_gaps.max():.2f}")
print(f"  fake gap >= our real gap in {(shuffled_gaps >= observed_gap).mean():.1%} of shuffles")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(shuffled_gaps, bins=40, color=GREY, edgecolor="white", linewidth=0.5,
        label="Gaps from randomly shuffled labels (n=1000)")
ax.axvline(observed_gap, color=ORANGE, linewidth=2.5,
           label=f"Our real gap (${observed_gap:.2f})")
ax.annotate("nothing random ever\ngot within $68 of this",
            xy=(observed_gap - 1, 30), xytext=(observed_gap - 30, 45),
            fontsize=9, color="#3d3d3a", ha="right", va="center",
            arrowprops=dict(arrowstyle="->", color="#3d3d3a", lw=1))
ax.set_title("The borough effect vs. what pure chance produces")
ax.set_xlabel("Gap between highest and lowest borough mean (USD)")
ax.set_ylabel("How often")
ax.set_ylim(0, 95)
ax.legend(loc="upper left")
plt.tight_layout(); plt.show()

**This is what a real finding looks like.**

The largest gap that chance produced in a thousand tries was **$24.91**, and the average was **$7.13**. Our observed gap is **$93.58** - nearly four times bigger than the best that randomness managed, and it never once came close. The p-value is **0.0%**: zero shuffles out of a thousand matched it.

Compare that to the layout of the chart. The orange line is not merely at the edge of the grey pile, it is in a different part of the plot entirely. When people say a result is "statistically significant", *this* is the picture they are describing.

So the borough effect is real. It is not an artifact of sorting, not an artifact of a zoomed axis, and not something we would see if we shuffled the labels. **We can act on it.**

Which is precisely the moment to be most careful, because the same machinery is about to produce something that looks identical and is worthless.

## The Same Dataset, One Layer Down

Boroughs are coarse. The obvious next move - the one an eager analyst makes on Monday morning - is to go finer: not "which borough is expensive" but **"which specific neighbourhood-and-room-type combination is the most expensive?"**

Same data. Same `groupby`. One more column. Watch what happens.

In [ ]:
combos = (clean.groupby(["neighbourhood", "room_type"])["price"]
          .agg(["count", "mean"]).sort_values("mean", ascending=False))

print("TOP 10 MOST EXPENSIVE NEIGHBOURHOOD / ROOM-TYPE COMBINATIONS")
print(combos.head(10).round(1).to_string())
print(f"\nTotal combinations: {len(combos)}")

In [ ]:
# The same top-10, drawn the way it would appear in a slide deck.
top10 = combos.head(10)
labels = [f"{n[:18]} - {r.replace(' home/apt', '')}" for n, r in top10.index]

fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.barh(range(len(top10)), top10["mean"], color=PINK, height=0.65)
ax.set_yticks(range(len(top10))); ax.set_yticklabels(labels, fontsize=9)
ax.invert_yaxis()
for i, (bar, v) in enumerate(zip(bars, top10["mean"])):
    ax.text(v + 8, i, f"${v:.0f}", va="center", fontsize=9, color="#3d3d3a")
ax.set_title("Top 10 most expensive neighbourhood / room-type combinations")
ax.set_xlabel("Average price per night (USD)")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

print("A confident chart. Now look at what it left out.")

That chart is a finding, a headline and a recommendation. *"Riverdale and Fort Wadsworth command $800 a night - four times the Manhattan average. We should target the outer-borough luxury segment."*

Now put the sample size back in.

In [ ]:
top10_with_n = combos.head(10).copy()
top10_with_n["count"] = top10_with_n["count"].astype(int)
print("THE SAME TABLE, WITH THE COLUMN THE CHART DROPPED")
print(top10_with_n.round(1).to_string())

print(f"\nOf {len(combos)} combinations, {(combos['count'] <= 5).sum()} are built on 5 listings or fewer.")
print(f"Combinations built on a SINGLE listing: {(combos['count'] == 1).sum()}")

**The top entry is one listing. And it is a shared room.**

Not a market segment, not a trend - a single Riverdale listing whose owner typed 800 into the price box for a *shared room*, which is the cheapest product on the platform everywhere else in the city. Its "average" is just its price, its rank is an accident, and it is sitting at the top of a chart labelled *most expensive* as if it were a fact about the Bronx.

The absurdity is the gift here. A shared room outranking every apartment in SoHo is obviously wrong, so you catch it. **The dangerous version of this mistake is the one that happens to look plausible** - and 55 of these combinations rest on a single listing, most of which will not announce themselves so helpfully.

It gets worse the further you look: **190 of the 540 combinations rest on five listings or fewer.** Those 190 are not a fringe - they are 35% of the table. Because tiny groups have wildly unstable averages, they are over-represented at the extreme ends of any ranking you build. The top of a sorted list is a magnet for small samples.

The contrast is the point. Both of these came from the same dataset, the same `groupby`, and charts that look equally authoritative:

| | Sample size | Verdict |
|---|---|---|
| Manhattan vs Bronx, $93.58 gap | 21,482 vs 1,089 | **Real** - beat 1000/1000 shuffles |
| Riverdale shared room, $800 | **1** | **Noise** - it is one listing |

Nothing about the *look* of a chart distinguishes them. Only the count does.

In [ ]:
# The chart that makes the trap visible: mean vs sample size, every combination.
fig, ax = plt.subplots(figsize=(9, 5))

ax.scatter(combos["count"], combos["mean"], s=18, alpha=0.55,
           color=BLUE, linewidths=0)
ax.axhline(clean["price"].mean(), color=GREY, linestyle="--", linewidth=1.2,
           label=f"overall mean (${clean['price'].mean():.0f})")
ax.set_xscale("log")
ax.set_xlabel("Number of listings in the combination (log scale)")
ax.set_ylabel("Average price (USD)")
ax.set_title("Why small groups win rankings: the funnel")

# Highlight the offenders the top-10 chart was made of.
tiny = combos.loc[combos["count"] <= 2]
ax.scatter(tiny["count"], tiny["mean"], s=26, color=ORANGE, linewidths=0,
           label=f"built on 1-2 listings (n={len(tiny)})")
ax.legend()
plt.tight_layout(); plt.show()

This shape has a name - a **funnel plot** - and once you have seen it you cannot unsee it.

On the left, where groups are tiny, prices are sprayed from near zero to $800. On the right, where groups are large, everything converges tightly onto the overall mean. The funnel is not telling you that small neighbourhoods are more expensive. **It is telling you that small samples are more variable**, which is a fact about arithmetic, not about New York.

Every extreme value in that chart - top and bottom - lives on the left-hand side. So when you sort by average and take the top 10, you are not selecting for "expensive". You are selecting for "small", and reading the resulting noise as a discovery.

### The name for this is p-hacking

There were **540 combinations** in that table. If you test enough of them, some will look extraordinary through luck alone - and it does not take many. This is the **multiple comparisons problem**: a 1-in-20 fluke stops being surprising once you have taken 540 draws.

The dangerous part is that nobody has to be dishonest for this to happen. You did not fabricate anything. You grouped by two columns, sorted, and took the top - three completely reasonable steps that together manufacture a finding out of nothing.

The defences are unglamorous and they work:

1. **Always show `count` next to `mean`.** Most of this section would have been impossible if the first chart had carried its sample sizes.
2. **Set a minimum group size before you look.** Deciding "at least 30 listings" *afterwards* is just another way of picking the answer you want.
3. **Shuffle-test the finding you are about to act on**, not the twenty you scrolled past.
4. **Correct for how many things you tested.** The formal versions - Bonferroni, false discovery rate - are the statistician's answer to exactly this, and they are the natural next thing to learn after this notebook. (See the **Statistics** pointer in the closing section.)

In [ ]:
# The honest version of the neighbourhood chart: enforce a minimum sample size,
# and show the uncertainty that remains.
MIN_LISTINGS = 50
solid = (clean.groupby(["neighbourhood", "room_type"])["price"]
         .agg(["count", "mean", "std"]))
solid = solid.loc[solid["count"] >= MIN_LISTINGS].copy()
solid["stderr"] = solid["std"] / np.sqrt(solid["count"])
solid = solid.sort_values("mean", ascending=False).head(10)

labels = [f"{n[:18]} - {r.replace(' home/apt', '')}" for n, r in solid.index]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(range(len(solid)), solid["mean"],
        xerr=solid["stderr"] * 1.96,          # 95% confidence interval
        color=BLUE, height=0.65, capsize=4,
        error_kw={"ecolor": "#3d3d3a", "elinewidth": 1.2})
ax.set_yticks(range(len(solid))); ax.set_yticklabels(labels, fontsize=9)
ax.invert_yaxis()
for i, (v, n) in enumerate(zip(solid["mean"], solid["count"])):
    ax.text(v + 22, i, f"${v:.0f}  n={int(n)}", va="center", fontsize=9, color="#3d3d3a")
ax.set_title(f"Most expensive combinations, minimum {MIN_LISTINGS} listings (95% CI)")
ax.set_xlabel("Average price per night (USD)"); ax.set_xlim(0, 430)
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

Same data, same question, a rule applied *before* looking - and now the answer is defensible.

Riverdale and Fort Wadsworth are gone. What is left is **Tribeca ($375, n=132), SoHo ($319, n=233), the Flatiron District, NoHo, the Theater District and Midtown** - expensive parts of Manhattan that an estate agent would have told you about, each resting on dozens or hundreds of listings. The error bars are visible but narrow enough to separate the top of the list from the bottom.

There is a real lesson in the boringness of that result. **A finding that matches what domain experts already believe is not a wasted analysis** - it is your method passing a calibration test. If your pipeline says SoHo is expensive, you can start to trust what it says about something you *cannot* check by intuition. That is precisely the "domain knowledge" leg from the very first cell of this notebook, doing its job.

### Three habits, earned rather than asserted

1. **Start bar charts at zero, or say clearly that you did not.** A zoomed axis manufactures drama for free.
2. **Show the sample size and the uncertainty.** `count` and error bars are what separate the Manhattan finding from the Fort Wadsworth one.
3. **Compare the gap between groups to the spread within groups** - and when it matters, shuffle-test it.

# Correlation - What Moves With What

We have been asking "what is the average of X, split by Y". The other question - **"which columns move together?"** - needs a different tool.

A **correlation** runs from -1 (perfect opposite) through 0 (no linear relationship) to +1 (perfect match). One number per pair of columns, and a heatmap shows all of them at once.

In [ ]:
numeric_cols = ["price", "minimum_nights", "number_of_reviews",
                "reviews_per_month", "availability_365", "calculated_host_listings_count"]
corr = clean[numeric_cols].corr()

short = ["price", "min_nights", "n_reviews", "reviews/mo", "availability", "host_listings"]

fig, ax = plt.subplots(figsize=(7.5, 5.5))
sns.heatmap(corr, annot=True, fmt=".3f", cmap=DIV, center=0, vmin=-1, vmax=1,
            square=True, linewidths=2, linecolor="white",
            xticklabels=short, yticklabels=short,
            cbar_kws={"shrink": 0.75, "label": "correlation"}, ax=ax)
ax.set_title("Correlation between the numeric columns")
plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
plt.setp(ax.get_yticklabels(), rotation=0)
plt.tight_layout(); plt.show()

Read the colour first, the numbers second. The map is almost entirely pale - **nearly every pair is uncorrelated** - with one clear exception.

**What is real:**

* **`number_of_reviews` and `reviews_per_month`: 0.589.** Comfortably the strongest relationship in the table, and also the least interesting: they are two views of the same underlying quantity. A listing with many reviews accumulates them at some rate. This is a *structural* correlation - it tells you about how the columns were defined, not about Airbnb. Feeding both to a model gives one fact double the vote.

**What is near-zero, and worth stating out loud:**

* **`price` and `minimum_nights`: 0.024.** Essentially nothing.
* **`price` and `number_of_reviews`: -0.058.** Expensive places are *very slightly* less reviewed, and that coefficient is far too small to build anything on.
* **`price` and `availability_365`: 0.118.** Weak, but the largest price relationship in the table.

Here is the trap in this section, and it is a subtle one. **If you had run only this correlation matrix, you would have concluded there was no signal in the dataset** - every price row is a rounding error away from zero. But we have already *proved* by shuffle test that borough moves price enormously, and we have watched the map show price falling off with distance from Midtown.

Both are true, because **correlation only measures straight-line relationships between numeric columns.** Borough is text, so it cannot appear here at all. Location is a pair of coordinates whose effect is a radial gradient, not a line - `latitude` alone is meaningless because price rises going north into Midtown and falls again going north into Harlem.

**A blank correlation matrix is not evidence of absence.** It is evidence that nothing is *linear and numeric*, which is a much narrower claim than most people hear it as.

In [ ]:
# Before trusting a column, look at its extremes. This is where the bodies are buried.
print("The 10 largest minimum_nights values in the RAW data:")
print(airbnb.nlargest(10, "minimum_nights")[
    ["neighbourhood_group", "room_type", "price", "minimum_nights", "number_of_reviews"]
].to_string(index=False))

print(f"\nListings requiring more than a YEAR: {(airbnb['minimum_nights'] > 365).sum()}")
print(f"Listings requiring more than a MONTH: {(airbnb['minimum_nights'] > 30).sum():,}"
      f"  ({(airbnb['minimum_nights'] > 30).mean():.1%})")
print(f"\nListings priced at $0: {(airbnb['price'] == 0).sum()}")
print(airbnb.loc[airbnb["price"] == 0,
                 ["neighbourhood_group", "room_type", "price", "availability_365"]].head().to_string(index=False))

Two genuine data-quality findings, and neither of them is the sort of thing a summary statistic would have surfaced.

**`minimum_nights` goes up to 1250.** That is three and a half years. Whatever those listings are - a mis-typed field, a landlord using Airbnb as a letting board, someone gaming the search filters - they are **not short-term rentals**, and short-term rentals are what this dataset is supposed to be about. Fourteen listings demand more than a year, and 747 - about 1.5% - demand more than a month, which is where New York's 30-day rule starts to show up in the data.

**Eleven listings are priced at $0.** A free night in New York is not a price point; it is a broken record. If those had stayed in, they would have quietly dragged down every average we computed, and no chart would have shown you why.

This is the "investigate before you trust it" habit, and the mechanics are simple: **`.nlargest()` and `.nsmallest()` on every column you plan to use.** The extremes of a column are where impossible values hide, and impossible values are far more damaging than missing ones - a gap announces itself, whereas a 1250 just quietly becomes part of your average.

Both of these are why the cleaning cell earlier removed what it removed. Not ritual. A specific response to specific things we went and looked for.

# Predicting Price

Everything so far has been analysis: describing what *is* in the data. Modelling asks a different question - **given what I know about a listing, can I predict something I do not know?**

The question: **can we predict a listing's price from its borough, room type, minimum nights, availability, review activity and host size?**

Three rules before we touch a model, and they are the whole discipline in miniature:

1. **Split the data.** Train on one part, evaluate on a part the model has never seen. A model graded on data it memorised will always look brilliant, in the same way a student who has seen the exam paper always looks clever. (`Intro_to_Computer_Science_and_AI.ipynb` covers this properly under overfitting.)
2. **Always have a baseline.** A score means nothing alone. Beating a stupid model is the minimum bar for claiming you learned anything, and the stupid model here is "ignore every feature and guess the average price".
3. **Decide what counts as success before you look.** Otherwise every number becomes evidence for whatever you already wanted.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score, mean_absolute_error

features = ["neighbourhood_group", "room_type", "minimum_nights",
            "availability_365", "number_of_reviews", "reviews_per_month",
            "calculated_host_listings_count"]

# Models need numbers. get_dummies turns each category into 0/1 columns.
# drop_first=True avoids one redundant column per category (they sum to 1).
X = pd.get_dummies(clean[features], drop_first=True)
y = clean["price"]

print("Original columns:", len(features), "-> after encoding:", X.shape[1])
print("Encoded columns:", list(X.columns))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print(f"\nTraining on {len(X_train):,} listings, testing on {len(X_test):,} unseen listings.")

In [ ]:
results = {}

# The baseline: ignore every feature, always guess the mean training price.
baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
linear = LinearRegression().fit(X_train, y_train)
forest = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1).fit(X_train, y_train)

print(f"{'Model':<24}{'Train R2':>10}{'Test R2':>10}{'Test MAE':>11}")
print("-" * 55)
for name, model in [("Baseline (guess mean)", baseline),
                    ("Linear Regression", linear),
                    ("Random Forest", forest)]:
    tr = r2_score(y_train, model.predict(X_train))
    te = r2_score(y_test, model.predict(X_test))
    mae = mean_absolute_error(y_test, model.predict(X_test))
    results[name] = {"train_r2": tr, "test_r2": te, "mae": mae}
    print(f"{name:<24}{tr:>10.4f}{te:>10.4f}{mae:>11.2f}")

**This time there is something there.** Note how different this feels from a result you have to explain away.

**R² is the share of price variation the model explains.** 1.0 is perfect; 0.0 means "no better than always guessing the average". The baseline scores **-0.0003**, which is the sanity check working: it knows nothing, so it explains nothing.

**Why can R² go negative?** Because "guess the average" is not the floor - it is the *reference point*, and you can do worse than it. A model that confidently predicts $400 for a $60 room is further from the truth than someone who shrugged and said $152. R² below zero means your model is actively worse than not having one.

**Linear Regression reaches 0.3225** - it explains about a third of the variation in price, using nothing but borough, room type and a few counts. In a messy human market like housing, a third is a genuinely useful model, not a disappointing one. Prices also depend on photos, furnishings, the actual street, the host's writing and the season, none of which are in these seven columns.

**Mean Absolute Error is the number to quote to a non-specialist.** The baseline is off by **$76.92** on a typical listing; the linear model by **$56.39**. That is a $20 improvement per listing in units anyone can picture, and unlike R², nobody has to be told what it means.

**Random Forest edges ahead on test R² (0.3343) and has the best MAE ($54.98)** - so on the held-out data, it wins. But look at the train column before you celebrate.

In [ ]:
# The train/test gap is the diagnostic. Draw it rather than squinting at four numbers.
names = list(results.keys())
train_scores = [results[n]["train_r2"] for n in names]
test_scores = [results[n]["test_r2"] for n in names]
yy = np.arange(len(names)); h = 0.36

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.barh(yy - h/2, train_scores, height=h, color=GREY, label="Train R2 (data it learned from)")
ax.barh(yy + h/2, test_scores, height=h, color=BLUE, label="Test R2 (unseen data)")
for i, (tr, te) in enumerate(zip(train_scores, test_scores)):
    ax.text(max(tr, 0) + 0.012, i - h/2, f"{tr:.3f}", va="center", fontsize=9, color="#3d3d3a")
    ax.text(max(te, 0) + 0.012, i + h/2, f"{te:.3f}", va="center", fontsize=9, color="#3d3d3a")
ax.set_yticks(yy); ax.set_yticklabels(names)
ax.invert_yaxis()
ax.axvline(0, color="#3d3d3a", linewidth=1)
ax.set_xlabel("R2  (0 = no better than guessing the mean)")
ax.set_title("Train vs test: the gap IS the overfitting")
ax.set_xlim(-0.05, 0.95); ax.legend(loc="lower right")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

gap = results["Random Forest"]["train_r2"] - results["Random Forest"]["test_r2"]
print(f"Random Forest train-test gap: {gap:.4f}")
print(f"Linear Regression gap:        "
      f"{results['Linear Regression']['train_r2'] - results['Linear Regression']['test_r2']:.4f}")

That chart is the most important one in this section, and it is worth more than the leaderboard.

**The Random Forest scores 0.8004 on data it trained on and 0.3343 on data it has not seen.** It looks like it explains 80% of price; it actually explains 33%. That gap of **0.47** is **overfitting**, and it is not a rounding error - it is the model memorising individual listings rather than learning the market.

**Linear Regression's gap is -0.03** - it scores *slightly better* on the test set than the training set, which is normal random variation and the signature of a model that is not memorising anything.

So which model wins? The forest, narrowly, on the metric that counts (test performance). But it wins by 0.012 R² while carrying an enormous memorisation habit, and that has consequences you do not see in this table: it will age worse as the market shifts, it is far harder to explain to a stakeholder, and its confidence is unearned.

**The honest summary is "the forest is marginally better and much less trustworthy."** If a colleague showed you only that 0.8004, you would have believed in a model twice as good as the one that exists. Reporting both columns is not pedantry - it is the entire defence against fooling yourself.

In [ ]:
# What did the model actually learn? Two views of the same forest.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

# Left: feature importance - which columns the forest leaned on.
imp = pd.Series(forest.feature_importances_, index=X.columns).sort_values()
axes[0].barh(imp.index, imp.values, color=BLUE, height=0.65)
for i, v in enumerate(imp.values):
    axes[0].text(v + 0.004, i, f"{v:.3f}", va="center", fontsize=8, color="#3d3d3a")
axes[0].set_title("What the forest leaned on")
axes[0].set_xlabel("Feature importance"); axes[0].set_xlim(0, 0.28)
axes[0].grid(axis="y", visible=False)

# Right: predicted vs actual. A perfect model would sit on the diagonal.
pred = forest.predict(X_test)
axes[1].scatter(y_test, pred, s=5, alpha=0.25, color=BLUE, linewidths=0)
lims = [0, 1000]
axes[1].plot(lims, lims, color=ORANGE, linewidth=1.8, label="perfect prediction")
axes[1].set_xlim(lims); axes[1].set_ylim(lims)
axes[1].set_xlabel("Actual price (USD)"); axes[1].set_ylabel("Predicted price (USD)")
axes[1].set_title("Predicted vs actual (Random Forest)")
axes[1].legend(loc="upper left")

plt.tight_layout(); plt.show()

**Left: what the forest used.** `availability_365` (0.221) and `room_type_Private room` (0.218) lead, followed by the review columns. Room type being near the top matches the analysis - whole apartments cost more than rooms, everywhere.

But be careful with this chart, because it is the most over-interpreted output in applied machine learning. **Importance is not causation, and it is not even effect size.** `availability_365` scoring highest does not mean listing your flat for more days raises its price. It means that column helped the forest split the data - and availability is tangled up with whether a listing is professionally managed, which is tangled up with price.

**Right: predicted vs actual.** A perfect model would lay every point on the orange diagonal. Two things stand out:

* The cloud slopes the right way - predictions rise with actual price, which is the model working.
* **It is badly squashed vertically.** For listings that actually cost $600-1000, predictions cluster around $200-300. The model systematically **under-predicts expensive listings**, because what makes a place cost $800 - the view, the address, the interior - is nowhere in our seven columns. It regresses toward the middle because the middle is the safest guess it can make from what it knows.

That is a far more useful conclusion than the R². It tells you exactly where the model fails and exactly what data you would need to fix it.

In [ ]:
# Residuals: where the errors live, and whether they are patterned or random.
residuals = y_test - pred

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(residuals, bins=60, color=BLUE, edgecolor="white", linewidth=0.4)
axes[0].axvline(0, color=ORANGE, linewidth=2)
axes[0].set_title("Distribution of errors (actual - predicted)")
axes[0].set_xlabel("Error (USD)"); axes[0].set_ylabel("Listings")

sc = axes[1].scatter(pred, residuals, s=5, alpha=0.25, c=residuals,
                     cmap=DIV, vmin=-400, vmax=400, linewidths=0)
axes[1].axhline(0, color="#3d3d3a", linewidth=1.5)
axes[1].set_title("Errors vs predicted price")
axes[1].set_xlabel("Predicted price (USD)"); axes[1].set_ylabel("Error (USD)")
plt.colorbar(sc, ax=axes[1], shrink=0.8, label="Error (USD)")

plt.tight_layout(); plt.show()

print(f"Mean error:   ${residuals.mean():7.2f}   (near zero = unbiased on average)")
print(f"Median error: ${residuals.median():7.2f}")
print(f"Worst under-prediction: ${residuals.max():.0f}")
print(f"Worst over-prediction:  ${residuals.min():.0f}")

**Left:** errors cluster around zero with a long right tail. The model is roughly unbiased on average - it is not systematically charging everyone too much or too little.

**Right:** and here is the pattern that the average hides. The errors **fan out** as predicted price rises, and the whole cloud tilts. At the cheap end the model is tight; at the expensive end it is wrong by hundreds, almost always in the same direction (under-predicting).

A residual plot should ideally look like a shapeless, even band around zero. **Structure in the residuals means there is signal the model failed to capture** - and this structure says the same thing the predicted-vs-actual chart did: our features describe ordinary listings well and luxury listings badly.

**This is the correct place to stop and be honest.** The model works, it beats its baseline convincingly, and it has a clearly-described failure mode. That is a real result. What would *not* be a real result is quoting 0.80, showing the leaderboard, and skipping both diagnostic charts.

# A Second Dataset, Same Discipline

Everything above was one dataset. The risk in that is subtle: you can learn the *dataset* instead of learning the *method*, and never notice until you meet a new one.

So here is a new one, from a completely different domain: **114,000 Spotify tracks**, each with audio features (danceability, energy, valence, tempo, acousticness) computed by Spotify's own analysis, plus a genre label and a popularity score.

Two questions, deliberately chosen:

* **Case A** - can we predict a track's **genre** from its audio features?
* **Case B** - can we predict a track's **popularity** from the same features?

They sound equally reasonable. They are not, and the difference is the whole point of this section. **A new dataset does not inherit the trust you built with the last one.** You run the same checks, every time, from scratch.

In [ ]:
spotify = load_hf_dataset("maharshipandya/spotify-tracks-dataset", "spotify.parquet")
assert spotify is not None, "Spotify data failed to load - see the message above."

print("Shape:", spotify.shape)
print("\nGenres:", spotify["track_genre"].nunique())

# "How many missing values" is the wrong question. "Where are they" is the right one.
missing_sp = spotify.isnull().sum()
print("\nMissing values, by column:")
print(missing_sp[missing_sp > 0] if missing_sp.sum() else "  none")
print("Rows with any missing value:", spotify.isnull().any(axis=1).sum())

spotify.head(3)

In [ ]:
AUDIO = ["danceability", "energy", "valence", "tempo", "acousticness",
         "instrumentalness", "speechiness", "liveness", "loudness", "duration_ms"]

print(spotify[AUDIO + ["popularity"]].describe().round(3).to_string())

## First Look at the Spotify Data

The first look pays off immediately, as it always does.

**114 genres.** That is a lot of classes for a first classification problem - with 114 labels, even a good model looks bad, and the confusion matrix becomes unreadable. We will pick **8 clearly different genres** instead. That is a modelling convenience, and the honest way to handle it is to *say so out loud* rather than quietly present a 114-class problem as an 8-class one.

**Three missing values in 114,000 rows x 21 columns.** Effectively nothing - and worth thirty seconds to identify rather than wave past. They are all in *one* row, which is missing its `artists`, `album_name` and `track_name`. Every audio feature is present. So the gap is in the metadata, not the measurements, and nothing we plan to model is affected.

That is the whole check: not "are there missing values" but "**where are they, and do they touch what I am about to use?**" A dataset can be 99.99% complete and still be missing exactly the column you need.

Note also what this tidiness does *not* buy you. A spotless table is sometimes a warning sign in itself - real human-entered data has gaps, so no gaps at all can mean the data was generated rather than collected. Here the explanation is mundane: these are machine-computed audio features, so they exist by construction. But **clean does not mean trustworthy**, as Case B is about to demonstrate rather forcefully.

**`popularity` runs 0-100 with a median of 35.** A real spread, so there is something to predict.

## Case A - Can Audio Features Predict Genre?

We narrow to eight genres that a human would never mix up, then do the thing this notebook has drilled into us: **look at the data before modelling it.**

In [ ]:
GENRES = ["acoustic", "black-metal", "classical", "hip-hop",
          "jazz", "reggae", "sleep", "techno"]
subset = spotify.loc[spotify["track_genre"].isin(GENRES)].copy()
print(f"{len(subset):,} tracks across {subset['track_genre'].nunique()} genres "
      f"({subset['track_genre'].value_counts().unique()} tracks each - perfectly balanced)")

# Do the features even look different by genre? Chart it BEFORE modelling.
fig, axes = plt.subplots(1, 4, figsize=(14, 3.8))
for ax, feat in zip(axes, ["energy", "acousticness", "danceability", "instrumentalness"]):
    sns.boxplot(data=subset, y="track_genre", x=feat, ax=ax,
                order=GENRES, color=BLUE, showfliers=False,
                linewidth=1, width=0.65)
    ax.set_title(feat); ax.set_ylabel(""); ax.set_xlabel("")
    if ax is not axes[0]:
        ax.set_yticklabels([])
plt.tight_layout(); plt.show()

**Look before you model.** These four charts are the reason Case A is going to work, and you can read the answer off them without training anything.

* **`energy`**: black-metal and techno sit high; sleep and classical sit near zero. Almost no overlap between those extremes.
* **`acousticness`**: classical and acoustic are pinned near 1.0; techno and black-metal near 0. This single feature nearly separates half the genres on its own.
* **`instrumentalness`**: sleep, classical and techno are instrumental; hip-hop and reggae have vocals.

The boxes are in *different places* for different genres, and mostly do not overlap. **That is what signal looks like before a model touches it.** Compare it with the price-vs-numeric correlations earlier, which were flat, and you have a reliable habit: if a well-chosen chart cannot see any difference between your groups, a model will usually struggle too - and if the chart *can*, you have good reason to expect the model to find it.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

Xg = subset[AUDIO]
yg = subset["track_genre"]

# stratify keeps every genre equally represented in both halves.
Xg_train, Xg_test, yg_train, yg_test = train_test_split(
    Xg, yg, test_size=0.2, random_state=42, stratify=yg)

dummy_clf = DummyClassifier(strategy="most_frequent").fit(Xg_train, yg_train)
rf_clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(Xg_train, yg_train)

base_acc = accuracy_score(yg_test, dummy_clf.predict(Xg_test))
test_acc = accuracy_score(yg_test, rf_clf.predict(Xg_test))
train_acc = accuracy_score(yg_train, rf_clf.predict(Xg_train))

print(f"{'Model':<28}{'Train acc':>11}{'Test acc':>11}")
print("-" * 50)
print(f"{'Baseline (most frequent)':<28}{'-':>11}{base_acc:>11.3f}")
print(f"{'Random Forest':<28}{train_acc:>11.3f}{test_acc:>11.3f}")
print(f"\nThe baseline is 1/8 = 0.125 because the 8 genres are balanced.")
print(f"The forest is {test_acc / base_acc:.1f}x better than guessing.")

In [ ]:
cm = confusion_matrix(yg_test, rf_clf.predict(Xg_test), labels=GENRES)

fig, ax = plt.subplots(figsize=(7.5, 6.5))
sns.heatmap(cm, annot=True, fmt="d", cmap=SEQ, square=True,
            xticklabels=GENRES, yticklabels=GENRES,
            linewidths=1.5, linecolor="white",
            cbar_kws={"shrink": 0.7, "label": "tracks"}, ax=ax)
ax.set_xlabel("Predicted genre"); ax.set_ylabel("Actual genre")
ax.set_title("Where the classifier is right - and what it confuses")
plt.tight_layout(); plt.show()

**79.2% accurate against a 12.5% baseline - more than six times better than guessing.** This is real signal, and the boxplots told us to expect it.

But the confusion matrix is the more interesting artifact, because **the mistakes are musically sensible**. The bright diagonal is the model getting it right; the off-diagonal cells are where it slips, and they cluster exactly where a human would slip:

* **hip-hop ↔ reggae** is by far the biggest confusion - 52 hip-hop tracks predicted as reggae, 46 the other way. Two vocal-led genres built on strong rhythm sections, and historically one grew out of the other.
* **jazz → acoustic** (32 tracks) and **acoustic → jazz** (18) - both acoustic, moderate energy, live instrumentation.
* **classical ↔ sleep** - 12 tracks each way. Quiet, instrumental, low energy. Plenty of sleep playlists *are* classical.

And look at what almost never gets confused: **black-metal and sleep share a single track between them in either direction.** Nothing about a black metal record resembles a sleep playlist, and the model knows it.

**A model whose errors make sense is a model you can reason about.** If the confusion had been uniform - every genre equally mistaken for every other - that would suggest the model learned nothing and got lucky on the easy classes. Instead it learned something close to how the genres actually sound.

Note the train accuracy too: **0.986 against 0.792 on test**. That is our overfitting friend again, in a classification setting. The forest has memorised a good chunk of the training set. It still generalises well, but the honest headline is 0.792, not 0.986 - and you only know that because you split the data.

## Case B - The Same Features, A Different Question

Now the question that sounds equally reasonable: **can these audio features predict how popular a track is?**

The pitch writes itself. If danceability and energy predict popularity, a label could screen demos, a playlist team could pick hits, and someone would build a startup on it.

Same dataset. Same features. Same train/test discipline. Run it.

In [ ]:
Xp = spotify[AUDIO]
yp = spotify["popularity"]

Xp_train, Xp_test, yp_train, yp_test = train_test_split(
    Xp, yp, test_size=0.2, random_state=42)

pop_base = DummyRegressor(strategy="mean").fit(Xp_train, yp_train)
pop_lin = LinearRegression().fit(Xp_train, yp_train)
pop_rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1).fit(Xp_train, yp_train)

print(f"{'Model':<24}{'Train R2':>10}{'Test R2':>10}")
print("-" * 45)
for name, m in [("Baseline (mean)", pop_base), ("Linear Regression", pop_lin),
                ("Random Forest", pop_rf)]:
    print(f"{name:<24}{r2_score(yp_train, m.predict(Xp_train)):>10.4f}"
          f"{r2_score(yp_test, m.predict(Xp_test)):>10.4f}")

Stop and read that before scrolling.

**Linear Regression: 0.0167.** Basically nothing - the linear relationship between audio features and popularity is negligible, which is roughly what you would expect.

**Random Forest: 0.5522 on the test set.** It explains over half the variation in popularity. On held-out data. With a proper train/test split.

By every rule this notebook has taught you so far, that is a genuine result. We split the data. We had a baseline. We beat it by an enormous margin on data the model never saw. The overfitting gap (0.91 train vs 0.55 test) is large but we have seen worse, and the test score is what counts.

**This is the trap. It has passed every check we have run so far, and it is still wrong.**

Before reading on, sit with the question the rest of this notebook has been training you to ask: *what would have to be true of this dataset for that number to be fake?*

In [ ]:
# The check the Airbnb data taught us: look at the data itself, not just the score.
print("Rows in the dataset:      ", f"{len(spotify):,}")
print("Unique track_id values:   ", f"{spotify['track_id'].nunique():,}")
print("Duplicated track_id rows: ", f"{spotify.duplicated('track_id').sum():,}")

# What does a duplicate actually look like?
dupe_ids = spotify.loc[spotify.duplicated("track_id", keep=False), "track_id"]
example = spotify.loc[spotify["track_id"] == dupe_ids.iloc[0]]
print("\nOne track, appearing more than once:")
print(example[["track_name", "artists", "track_genre", "popularity", "energy"]].to_string(index=False))

**There it is.** 114,000 rows, but only **89,741 unique tracks** - **24,259 duplicated rows**.

The same song appears several times, once per genre it was filed under. Identical audio features. Identical popularity. Different `track_genre` label.

Now think about what `train_test_split` did with that. It shuffled **rows**, not **songs**. So a track filed under three genres could land twice in training and once in test - and at test time the forest was handed a row it had already memorised, popularity and all.

**It was not predicting popularity. It was looking up songs it had already seen.**

This is **data leakage**, and it is the single most common way a machine learning result turns out to be worthless in practice. The model performs superbly in evaluation and collapses in production, because in production the songs really are new.

The fix is to split on the *entity* you actually care about, not on rows.

In [ ]:
# Dedupe by track, THEN split. One row per song, so no song can be in both halves.
unique_tracks = spotify.drop_duplicates("track_id")
print(f"{len(spotify):,} rows -> {len(unique_tracks):,} unique tracks")

Xu = unique_tracks[AUDIO]
yu = unique_tracks["popularity"]
Xu_train, Xu_test, yu_train, yu_test = train_test_split(
    Xu, yu, test_size=0.2, random_state=42)

u_lin = LinearRegression().fit(Xu_train, yu_train)
u_rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1).fit(Xu_train, yu_train)

print(f"\n{'Model':<24}{'Train R2':>10}{'Test R2':>10}")
print("-" * 45)
print(f"{'Linear Regression':<24}{r2_score(yu_train, u_lin.predict(Xu_train)):>10.4f}"
      f"{r2_score(yu_test, u_lin.predict(Xu_test)):>10.4f}")
print(f"{'Random Forest':<24}{r2_score(yu_train, u_rf.predict(Xu_train)):>10.4f}"
      f"{r2_score(yu_test, u_rf.predict(Xu_test)):>10.4f}")

In [ ]:
# The whole story of Case B in one chart.
labels = ["Random Forest\n(duplicates left in)", "Random Forest\n(deduplicated)"]
test_vals = [0.5522, r2_score(yu_test, u_rf.predict(Xu_test))]
train_vals = [0.9066, r2_score(yu_train, u_rf.predict(Xu_train))]

xx = np.arange(2); w = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(xx - w/2, train_vals, width=w, color=GREY, label="Train R2")
ax.bar(xx + w/2, test_vals, width=w, color=BLUE, label="Test R2")
for i, (tr, te) in enumerate(zip(train_vals, test_vals)):
    ax.text(i - w/2, tr + 0.015, f"{tr:.3f}", ha="center", fontsize=9, color="#3d3d3a")
    ax.text(i + w/2, te + 0.015, f"{te:.3f}", ha="center", fontsize=9, color="#3d3d3a")
ax.set_xticks(xx); ax.set_xticklabels(labels)
ax.set_ylabel("R2"); ax.set_ylim(0, 1.0)
ax.set_title("The same model, the same features - one line of cleaning apart")
ax.legend(loc="upper right")
ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

**Test R² falls from 0.5522 to 0.2434 - more than half the "performance" was leakage.**

Two lessons, and the second is the one people miss.

**First: the headline number was inflated by a factor of more than two**, purely because of how the split was done. Nobody cheated. `train_test_split` is the correct function; `random_state=42` is the standard incantation; the code is what every tutorial shows. The bug was in the *assumption* that a row is a song.

**Second: what remains is still not what it appears.** The deduplicated forest scores 0.2434 on test against **0.8710 on train** - a gap of 0.63, far worse than anything the Airbnb model did. And Linear Regression manages **0.0250**, which is nearly nothing. Put those together and the picture is: there is no meaningful *linear* relationship between how a song sounds and how popular it is, and a heavily-overfit forest scrapes out a modest amount by memorising fine-grained combinations that will not transfer to next year's music.

**The honest verdict on Case B: audio features tell you very little about popularity.** Which is not surprising if you think about the world rather than the table - popularity is driven by the artist's existing fame, marketing budget, playlist placement, TikTok, and release timing. None of that is in `danceability`.

### The connection back to the Airbnb data

Case A and Case B use the same dataset, the same features, and the same code. One is a solid result; the other is an artifact that would have shipped.

What separated them was not a cleverer model. It was **going back to look at the data when a number seemed too good** - counting rows, counting unique IDs, printing one example. Exactly what we did when a $800 shared room topped the Airbnb rankings.

**A new dataset does not earn your trust automatically. You check every time.** The Spotify data had no missing values and looked immaculate, and it still had a trap in it big enough to double a headline result.

# What We Covered

* **NumPy** - the `ndarray`, and *why* it is fast rather than just that it is. Boolean masks, and the `axis` rule that the collapsed dimension is the one that disappears.
* **Pandas** - Series and DataFrames, loading real data with caching and error handling, `.loc` vs `.iloc`, filtering, and split-apply-combine with `groupby`.
* **Cleaning as judgement, not ritual** - a missing `reviews_per_month` meant *zero reviews*, so we filled it with 0. A reflexive `dropna()` would have deleted 10,052 listings, a fifth of New York, for no reason. Three kinds of missing, three different answers.
* **EDA** - `.describe()` flagged a $0 minimum price, a 10,000 maximum and a mean well above the median. All three turned out to matter.
* **Visualising** - histograms and the log scale, box plots showing overlap that bar charts hide, stacked bars revealing that Manhattan's premium is partly product mix, and 48,631 dots at their coordinates drawing a recognisable map of New York with price fading outward from Midtown.
* **When a chart does and does not lie** - the borough effect beat **1000 out of 1000** random shuffles ($93.58 real vs $24.91 best fake), so we acted on it. One `groupby` deeper, a $800 *shared room* built on a **single listing** topped the rankings - and 190 of 540 combinations rested on five listings or fewer. Same data, same method, opposite verdicts. Only `count` told them apart.
* **Multiple comparisons** - test 540 combinations and something will look extraordinary by luck. Set your minimum sample size *before* you look.
* **Correlation is narrow** - every numeric correlation with price was near zero, while borough moved price by 2.1x. A blank correlation matrix is not evidence of absence; it is evidence that nothing is *linear and numeric*.
* **Modelling** - baseline, split, evaluate. Linear Regression explained 32% of price variation and cut typical error from $76.92 to $56.39. The Random Forest edged it on test (0.3343) while scoring 0.8004 on train - better *and* less trustworthy. Residual plots showed exactly where it failed: luxury listings, whose value lives in columns we do not have.
* **Leakage** - a Spotify model scored 0.5522 predicting popularity and passed every check, until we counted unique track IDs and found 24,259 duplicate rows spanning the split. Deduplicated, it scored 0.2434. The genre classifier next door was real (79.2% against a 12.5% baseline), which is precisely why the popularity result was so easy to believe.

**The thread.** The tools are the easy part - you can look up `groupby` any time. The skill is scepticism aimed at your own results, especially the ones you like. This notebook found real, defensible effects *and* two convincing artifacts, using identical code. The difference was never a better algorithm. It was going back to look at the data when a number seemed too good.

# Homework

**Challenge 1: Finish the "Try it yourself" cells**
There are two above - the NumPy `axis` one and the `groupby` on `room_type`. Do those first.

**Challenge 2: Find a dataset**
Go to [Kaggle](https://www.kaggle.com/datasets) and pick something you actually care about - sport, music, games, climate. Interest carries you through the boring parts.

To load it in Colab, either upload it via the file browser on the left (then read `/content/your_file.csv`), or find it on the [Hugging Face Hub](https://huggingface.co/datasets) and download it programmatically like we did above. The second way means your notebook re-runs anywhere.

**Challenge 3: Run the first look**
`.shape`, `.head()`, `.info()`, `.describe()`. Then write a markdown cell answering: how many rows? Which columns have missing values, and *why* are they missing? Anything in `.describe()` that looks impossible - a negative age, a price of zero, a minimum stay of 1250 nights?

**Challenge 4: Clean it - but only what needs cleaning**
For every column with gaps, decide which kind of missing it is: genuinely unknown (impute), never happened (fill with zero), or broken upstream (investigate). Write down what you decided and why. Do not call `dropna()` as a ritual.

**Challenge 5: Five charts, five questions**
A histogram, a bar chart with sample sizes labelled, a box plot, a scatter, and one more. Each has to answer a question you wrote down *before* you drew it.

**Challenge 6: Try to break your own finding**
Pick your most interesting result and attack it:
* Does the bar chart's axis start at zero? Redraw it if not.
* How many rows is each group built on? Are any of them tiny?
* Run the shuffle test. How often does chance beat you?
* If you sorted and took the top N, how many combinations did you test to get there?

**Challenge 7: Model it, and check for leakage**
Build a baseline, then a model. Report train *and* test scores. Then ask the Case B question: **is anything in my features secretly a copy of the answer?** Are there duplicate rows that could span the split? Is one of my columns something I would not actually know at prediction time?

**Challenge 8: Report honestly**
Write a short markdown summary - **including anything that turned out to be noise**. A notebook that says "I expected X, tested it, and it was not there" is better work than one with five confident charts of nothing.

In [ ]:
# Challenge 2: load your dataset here.
# Option A - upload via the file browser on the left, then:
# my_df = pd.read_csv("/content/your_file.csv")

# Option B - reuse our loader (this way the notebook re-runs anywhere):
# my_df = load_hf_dataset("OWNER/DATASET-NAME", "mydata.parquet")

# TODO: your code here.

In [ ]:
# Challenge 3: the first look. shape, head, info, describe.
# TODO: your code here.

In [ ]:
# Challenge 4: for each column with gaps - unknown, never-happened, or broken?
# TODO: your code here.

In [ ]:
# Challenge 5: five charts, each answering a question you wrote down first.
# TODO: your code here.

In [ ]:
# Challenge 6: attack your best finding. Sample sizes, zeroed axis, shuffle test.
# TODO: your code here.

In [ ]:
# Challenge 7: baseline, model, train AND test scores - then hunt for leakage.
# TODO: your code here.

### Challenge 8 - write your honest summary here as markdown

*Replace this cell with what you found, what you expected, and what turned out to be noise.*

# Where to Go Next

* **Seaborn** - we used it for box plots and heatmaps here; it also does pair plots, regression plots and faceted grids in one line, with confidence intervals by default.
* **`Intro_to_Computer_Science_and_AI.ipynb`** - the theory under the modelling sections: overfitting, train/test splits, and why more model is not more answer.
* **`Intro_to_NLP.ipynb`** - the same workflow where the data is text instead of numbers.
* **Statistics** - the shuffle test is a **permutation test** in disguise, and the small-sample trap is the **multiple comparisons problem**. Learning the formal versions - t-tests, ANOVA, confidence intervals, Bonferroni correction and false discovery rate - turns the intuitions in this notebook into something you can defend rigorously.
* **Feature engineering** - our model never knew how far a listing was from Midtown, though the map showed that mattered enormously. Computing distance from a few landmarks would likely beat every model we built.